In [ ]:
from typing import Callable
from pipe import select

import numpy as np
import pandas as pd
from scipy.integrate import solve_ivp

import plotly.express as px
import plotly.graph_objects as go

In [553]:
# exact solution
def solution(t: np.ndarray, lambda1: float, lambda2: float, x0: np.ndarray):
    e1 = np.array([1., 0.])[:, None]
    e2 = np.array([0., 1.])[:, None]
    l = np.repeat(np.array([lambda1, lambda2])[:, None], t.shape[0], axis=1)
    sol = x0 * np.exp(l * t).T
    return sol

In [554]:
# building matrix A in our ODE x' = A @ x
def system_matrix(
    lambda1: float,
    lambda2: float,
    e1: np.ndarray,
    e2: np.ndarray
):
    L = np.diag([lambda1, lambda2])
    U = np.hstack([e1, e2])
    U /= np.linalg.norm(U, axis=0)
    A = (U @ L @ U.T)
    return A

In [555]:
# system params
systems = [
    {
        "lambda1": -35.,
        "lambda2": 0.5,
        "e1": np.array([1., 0])[:, None],
        "e2": np.array([0., 1.])[:, None],
    },
    {
        "lambda1": -10.,
        "lambda2": 0.6,
        "e1": np.array([1., 0])[:, None],
        "e2": np.array([0., 1.])[:, None],
    }
]

In [556]:
# system's matrices
sys_matr = list(systems | select(lambda d: system_matrix(**d)))
sys_matr

[array([[-35. ,   0. ],
        [  0. ,   0.5]]),
 array([[-10. ,   0. ],
        [  0. ,   0.6]])]

In [557]:
x0 = np.array([1e3, 1.])

In [558]:
T = 0.5
delta_t = 0.5e-1
t_space = np.linspace(0., T, int(T / delta_t))

In [559]:
# compute exact solutions
true_sol = list(
    systems |
    select(lambda d: solution(t_space, d["lambda1"], d["lambda2"], x0))
)

In [560]:
def euler(f: Callable, x0: np.ndarray, t_space: np.ndarray):
    # use functional programming
    traj = np.zeros([t_space.shape[0], x0.shape[0]])
    traj[0] = x0

    for i in range(1, t_space.shape[0]):
        traj[i] = traj[i-1] + (t_space[i] - t_space[i-1]) * f(traj[i-1], t_space[i-1])

    return traj

In [561]:
# solve sytems with euler
euler_sol = list(
    sys_matr |
    select(lambda A: euler(lambda x, t: A @ x, x0, t_space))
)

In [562]:
# solve sytems with RK23
RK23_sol = list(
    sys_matr |
    select(
        lambda A: solve_ivp(
            lambda t, x: A.dot(x),
            [0., T],
            x0,
            method="RK23",
            t_eval=t_space
        ).y.T
    )
)

In [563]:
RK23_sol[0].shape

(10, 2)

In [564]:
solutions = {
    "true": true_sol,
    "euler": euler_sol,
    "RK23": RK23_sol
}

In [565]:
fig = go.Figure()
for sol_name, sol in solutions.items():
    for i, sys_sol in enumerate(sol):
        fig.add_trace(
            go.Scatter(
                x=sys_sol[:, 0],
                y=sys_sol[:, 1],
                mode='lines',
                name=f"{sol_name}_{i}"
            )
        )
fig.add_trace(
    go.Scatter(
        x=[x0[0]],
        y=[x0[1]],
        name="start"
    )
)
fig.show()

Let's classify noisy trajectory from system 0 and measure accuracy

In [566]:
NUM_SAMPLES = 1000
sigmas = np.logspace(0., 5, 10)
# what system to classify
sys_to_cls = 0
num_traj_points = true_sol[sys_to_cls].shape[0]

In [567]:
sigmas

array([1.00000000e+00, 3.59381366e+00, 1.29154967e+01, 4.64158883e+01,
       1.66810054e+02, 5.99484250e+02, 2.15443469e+03, 7.74263683e+03,
       2.78255940e+04, 1.00000000e+05])

In [568]:
def cls_loss(sample_traj: np.ndarray, mean_traj: np.ndarray):
    return np.linalg.norm(sample_traj - mean_traj, ord=2, axis=1).mean()

In [569]:
accuracy_df = []
for sigma in sigmas:
    cur_accuracy_df = pd.DataFrame(
        {
            "accuracy": np.zeros([len(solutions)]),
            "method": ["true", "euler", "RK23"],
            "sigma": np.full([len(solutions)], sigma)
        }
    )
    for i in range(NUM_SAMPLES):
        sample_traj = true_sol[sys_to_cls] + sigma * np.random.randn(num_traj_points, 2)
        for sol_name, sol in solutions.items():
            sol_losses = list(
                range(2) |
                select(lambda sys_id: cls_loss(sample_traj, sol[sys_id]))
            )
            if sol_losses[sys_to_cls] < sol_losses[1 - sys_to_cls]:
                cur_accuracy_df.loc[
                    cur_accuracy_df["method"] == sol_name, 
                    "accuracy"
                ] += 1.
    cur_accuracy_df.loc[:, "accuracy"] /= NUM_SAMPLES
    accuracy_df.append(cur_accuracy_df)

In [570]:
accuracy_df = pd.concat(accuracy_df, axis=0)
accuracy_df

,accuracy,method,sigma
0,1.000,true,1.000000
1,0.000,euler,1.000000
2,1.000,RK23,1.000000
0,1.000,true,3.593814
1,0.000,euler,3.593814
2,1.000,RK23,3.593814
0,1.000,true,12.915497
1,0.000,euler,12.915497
2,1.000,RK23,12.915497
0,1.000,true,46.415888


In [ ]:
px.line(
    accuracy_df,
    x="sigma",
    y="accuracy",
    color="method",
    line_dash="method"
)